# Embed + Index — MedEmbed + ChromaDB

Turns `corpus/chunks/all_chunks.jsonl` into a searchable vector index. Fully local — no API keys, no rate limits.

**Stack**
- Embedding model: **`abhinand/MedEmbed-large-v0.1`** (1024-dim). A `bge-large-en-v1.5` fine-tuned on **clinical text from PubMed Central** using synthetic (query, positive, negative) triplets. On medical IR benchmarks (NFCorpus, PublicHealthQA, TRECCOVID, MedicalQARetrieval) it beats general embedders of similar size.
- Vector DB: **ChromaDB** (local, persistent, stores metadata natively).
- Similarity: **cosine** (embeddings are L2-normalized so cosine = dot product).

**Model caching (why you won't re-download every run)**  
HuggingFace caches downloaded model weights at `~/.cache/huggingface/hub/` by default (about **1.3 GB** for MedEmbed-large). First run downloads once; every subsequent run loads from disk in ~5 seconds. Do **not** delete that cache folder unless you need the disk space back.

**Inputs**
- `corpus/chunks/all_chunks.jsonl` (477 chunks, produced by `chunker.ipynb`)

**Outputs**
- `corpus/chunks/chunk_embeddings.npy` — raw embeddings (regenerable; gitignored)
- `corpus/chunks/chroma_db/` — ChromaDB persistent index (regenerable; gitignored)

In [1]:
import json
import os
import time
import numpy as np
from pathlib import Path

CHUNKS_PATH = "corpus/chunks/all_chunks.jsonl"
OUT_DIR = "corpus/chunks"
CHROMA_DIR = os.path.join(OUT_DIR, "chroma_db")
EMBEDDINGS_PATH = os.path.join(OUT_DIR, "chunk_embeddings.npy")

COLLECTION_NAME = "ckd_guidelines"
EMBED_MODEL_NAME = "abhinand/MedEmbed-large-v0.1"
EMBED_DIM = 1024                    # asserted after model loads
BATCH_SIZE = 32                     # embedding batch — CPU-friendly

# Instruction prefix on the QUERY side only (MedEmbed is a bge-style model,
# and bge-style models are trained with query-side instructions).
QUERY_INSTRUCTION = "Represent this medical question for retrieving relevant clinical guideline passages: "

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Model: {EMBED_MODEL_NAME}")
print(f"Output: {CHROMA_DIR}")

Model: abhinand/MedEmbed-large-v0.1
Output: corpus/chunks\chroma_db


## Step 1 — Load chunks

In [2]:
all_chunks = []
with open(CHUNKS_PATH, encoding="utf-8") as f:
    for line in f:
        all_chunks.append(json.loads(line))

print(f"Loaded {len(all_chunks)} chunks from {CHUNKS_PATH}")
print(f"Sample: {all_chunks[0]['chunk_id']} | {all_chunks[0]['token_count']} tokens | {all_chunks[0]['section_title'][:60]}")

Loaded 477 chunks from corpus/chunks/all_chunks.jsonl
Sample: kdigo_p14_c01 | 612 tokens | notice


## Step 2 — Load MedEmbed (from HF cache after first run)

In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from sentence_transformers import SentenceTransformer

t0 = time.time()
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
load_time = time.time() - t0

actual_dim = embed_model.get_sentence_embedding_dimension()
assert actual_dim == EMBED_DIM, f"Expected {EMBED_DIM}-dim, got {actual_dim}"

print(f"Model loaded in {load_time:.1f}s (from HF cache if not first run) | dim={actual_dim}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded in 5.0s (from HF cache if not first run) | dim=1024


## Step 3 — Embed all chunks (or reuse saved embeddings)

If `chunk_embeddings.npy` already exists and matches the current chunk count, we reuse it. Set `FORCE_REEMBED = True` to override.

In [4]:
FORCE_REEMBED = False

if not FORCE_REEMBED and os.path.exists(EMBEDDINGS_PATH):
    cached = np.load(EMBEDDINGS_PATH)
    if cached.shape == (len(all_chunks), EMBED_DIM):
        embeddings = cached
        print(f"Reused cached embeddings from {EMBEDDINGS_PATH} (shape {cached.shape})")
    else:
        print(f"Cache shape mismatch (was {cached.shape}, expected {(len(all_chunks), EMBED_DIM)}). Re-embedding.")
        embeddings = None
else:
    embeddings = None

if embeddings is None:
    texts = [c["text"] for c in all_chunks]
    t0 = time.time()
    embeddings = embed_model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,       # cosine = dot product after this
        convert_to_numpy=True,
    )
    print(f"Embedded {len(texts)} chunks in {time.time()-t0:.1f}s | shape {embeddings.shape}")
    np.save(EMBEDDINGS_PATH, embeddings)
    print(f"Saved to {EMBEDDINGS_PATH}")

Reused cached embeddings from corpus/chunks\chunk_embeddings.npy (shape (477, 1024))


## Step 4 — Load into ChromaDB

Fresh collection each run — deterministic and cheap. Model name + dim + similarity metric recorded in collection metadata so future work knows what built the index.

In [5]:
import chromadb

client = chromadb.PersistentClient(path=CHROMA_DIR)

if COLLECTION_NAME in [c.name for c in client.list_collections()]:
    client.delete_collection(COLLECTION_NAME)

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={
        "embed_model": EMBED_MODEL_NAME,
        "embed_dim": EMBED_DIM,
        "hnsw:space": "cosine",
        "query_instruction": QUERY_INSTRUCTION,
    },
)

def to_chroma_metadata(c):
    """Chroma requires flat, scalar metadata. Split page_range into two ints."""
    return {
        "document_name": c["document_name"],
        "source_url": c["source_url"],
        "page_number": c["page_number"],
        "page_start": c["page_range"][0],
        "page_end":   c["page_range"][1],
        "section_title": c["section_title"],
        "token_count": c["token_count"],
    }

BATCH = 200
for i in range(0, len(all_chunks), BATCH):
    batch = all_chunks[i:i + BATCH]
    collection.add(
        ids=[c["chunk_id"] for c in batch],
        embeddings=embeddings[i:i + BATCH].tolist(),
        documents=[c["text"] for c in batch],
        metadatas=[to_chroma_metadata(c) for c in batch],
    )

print(f"Loaded {collection.count()} vectors into '{COLLECTION_NAME}' at {CHROMA_DIR}")
print(f"Collection metadata: {collection.metadata}")

Loaded 477 vectors into 'ckd_guidelines' at corpus/chunks\chroma_db
Collection metadata: {'hnsw:space': 'cosine', 'query_instruction': 'Represent this medical question for retrieving relevant clinical guideline passages: ', 'embed_model': 'abhinand/MedEmbed-large-v0.1', 'embed_dim': 1024}


## Step 5 — Baseline retrieval test (Day-1 deliverable)

Deck exit criteria: *"question → top chunks → source page/section."*  
8 clinical questions across categories that Day 4 will formally evaluate.

In [6]:
TEST_QUESTIONS = [
    ("direct",       "What is the diagnostic threshold for albuminuria in CKD?"),
    ("direct",       "When should an SGLT2 inhibitor be started in a patient with CKD and type 2 diabetes?"),
    ("direct",       "What are the GFR categories G1 through G5?"),
    ("multi",        "What blood pressure target is recommended for adults with CKD and albuminuria?"),
    ("multi",        "Which drug class is first-line for CKD with hypertension and proteinuria?"),
    ("edge",         "How should potassium be monitored when starting a mineralocorticoid receptor antagonist?"),
    ("edge",         "How often should eGFR be checked in a CKD patient?"),
    ("out_of_scope", "What is the recommended treatment for acute appendicitis?"),
]


def retrieve(query: str, k: int = 3):
    q_emb = embed_model.encode(
        [QUERY_INSTRUCTION + query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    res = collection.query(query_embeddings=q_emb.tolist(), n_results=k)
    hits = []
    for i in range(len(res["ids"][0])):
        hits.append({
            "chunk_id":   res["ids"][0][i],
            "distance":   res["distances"][0][i],
            "similarity": 1 - res["distances"][0][i],
            "metadata":   res["metadatas"][0][i],
            "text":       res["documents"][0][i],
        })
    return hits


for category, q in TEST_QUESTIONS:
    print("=" * 80)
    print(f"[{category.upper()}] Q: {q}")
    print("=" * 80)
    for rank, h in enumerate(retrieve(q, k=3), 1):
        m = h["metadata"]
        print(f"  [{rank}] sim={h['similarity']:.3f}  {h['chunk_id']}")
        print(f"      {m['document_name']} — {m['section_title']} — p{m['page_number']}")
        print(f"      {h['text'].strip()[:200].replace(chr(10),' ')}...")
    print()

[DIRECT] Q: What is the diagnostic threshold for albuminuria in CKD?


  [1] sim=0.760  kdigo_p155_c16
      KDIGO 2024 CKD Guideline — Chapter 6: Research recommendations — p155
      In children and young adults with suspected or diagnosed CKD, what is the accuracy of the albumin-tocreatinine ratio (ACR) and protein-to-creatinine ratio (PCR) compared with 24-hour excretion of albu...
  [2] sim=0.759  kdigo_p155_c17
      KDIGO 2024 CKD Guideline — Chapter 6: Research recommendations — p155
      Population Adults and children (Continued on following page) methods for guideline development  Table 44| (Continued) Clinical questions and systematic review topics in PICOS format Chapter 1 Evaluati...
  [3] sim=0.755  kdigo_p83_c12
      KDIGO 2024 CKD Guideline — 2.2 Risk prediction in people with CKD — p83
      small fraction of the CKD population is at high risk for progression to kidney failure, those people with lower risks of progression to kidney failure may be effectively managed in primary care settin...

[DIRECT] Q: When should an SGLT2 inhibitor b

  [1] sim=0.769  nice_p51_c02
      NICE NG203 - Chronic kidney disease: assessment and management — Terms used in this guideline — p51
      Classification of CKD  CKD is classified according to estimated GFR (eGFR) and albumin:creatinine ratio (ACR)  (see table 1), using 'G' to denote the GFR category (G1 to G5, which have the same GFR  t...
  [2] sim=0.751  nice_p13_c02
      NICE NG203 - Chronic kidney disease: assessment and management — 1.2 Classification of CKD in adults — p13
      .73 m2)  Moderate risk  High risk  Very high risk  GFR category G3b: moderate to severe  reduction (30 to 44 ml/min/1.73 m2)  High risk  Very high risk  Very high risk  ACR category A1: normal to  mil...
  [3] sim=0.743  nice_p15_c02
      NICE NG203 - Chronic kidney disease: assessment and management — 1.3 Frequency of monitoring — p15
      progression is often non-linear)  • other risk factors, including heart failure, diabetes and hypertension  • changes to their treatment (such as renin–angioten

  [1] sim=0.730  kdigo_p91_c02
      KDIGO 2024 CKD Guideline — 3.2.1 Avoiding use of tobacco products — p91
      the KDIGO 2021 Clinical Practice Guideline for the Management of Blood Pressure in Chronic Kidney Disease21 and KDIGO 2022 Clinical Practice Guideline for Diabetes Management in Chronic Kidney Disease...
  [2] sim=0.724  nice_p23_c04
      NICE NG203 - Chronic kidney disease: assessment and management — 1.6 Pharmacotherapy — p23
      young people with CKD and diabetes (type 1 or 2), offer an ARB  or an ACE inhibitor (titrated to the highest licensed dose that they can tolerate) if  ACR is 3 mg/mmol or more. [2021]  1.6.11  For chi...
  [3] sim=0.718  nice_p51_c04
      NICE NG203 - Chronic kidney disease: assessment and management — Terms used in this guideline — p51
      This group of medicines does not include aldosterone  antagonists.  Recommendations for research  As part of the 2021 update, the guideline committee made 18 recommendations for  research on chronic k..

  [1] sim=0.794  kdigo_p81_c02
      KDIGO 2024 CKD Guideline — 2.1 Overview on monitoring for progression of CKD — p81
      Special considerations Pediatric considerations.  Monitoring of children in the peripubescent phase should be undertaken more frequently than the CKD stage–based recommended frequency of monitoring as...
  [2] sim=0.783  nice_p51_c13
      NICE NG203 - Chronic kidney disease: assessment and management — Terms used in this guideline — p51
      How the recommendations might affect practice  The recommendations are in line with current practice, so no additional resources should  be needed.  Return to recommendations  Frequency of monitoring ...
  [3] sim=0.775  kdigo_p99_c01
      KDIGO 2024 CKD Guideline — 3.7 Sodium-glucose cotransporter-2 inhibitors — p99
      eGFR <30 ml/min per 1.73 m2, compared with those who continue.508,509 In addition, a recent individual patient level data meta-analysis demonstrated a benefit in delaying KRT in patients with eGFR <30..

## Step 6 — Try your own query

Change the string below and re-run this cell only. No re-embedding needed.

In [7]:
MY_QUERY = "What blood pressure target should adults with CKD and high albuminuria aim for?"
TOP_K = 3

print(f'QUERY: "{MY_QUERY}"\n')
for rank, h in enumerate(retrieve(MY_QUERY, k=TOP_K), 1):
    m = h["metadata"]
    print(f"[{rank}] similarity={h['similarity']:.3f}")
    print(f"    chunk_id : {h['chunk_id']}")
    print(f"    document : {m['document_name']}")
    print(f"    section  : {m['section_title']}")
    print(f"    page     : {m['page_number']} (range {m['page_start']}-{m['page_end']})")
    print("    text:")
    print(h["text"][:500], "..." if len(h["text"]) > 500 else "")
    print("-" * 80)

QUERY: "What blood pressure target should adults with CKD and high albuminuria aim for?"



[1] similarity=0.839
    chunk_id : nice_p23_c02
    document : NICE NG203 - Chronic kidney disease: assessment and management
    section  : 1.6 Pharmacotherapy
    page     : 23 (range 23-28)
    text:
NICE's guideline on hypertension in adults recommends using clinic blood pressure for 
monitoring response to lifestyle changes or medical treatment (see recommendation 
1.4.15).

1.6.1 
In adults with CKD and an ACR under 70 mg/mmol, aim for a clinic systolic blood 
pressure below 140 mmHg (target range 120 to 139 mmHg) and a clinic diastolic 
blood pressure below 90 mmHg. [2021] 
1.6.2 
In adults with CKD and an ACR of 70 mg/mmol or more, aim for a clinic systolic 
blood pressure below 130 mmH ...
--------------------------------------------------------------------------------
[2] similarity=0.810
    chunk_id : kdigo_p97_c01
    document : KDIGO 2024 CKD Guideline
    section  : 3.4 Blood pressure control
    page     : 97 (range 97-97)
    text:
≥3 mm Hg over 24 hours while on a hi

## Summary

**Day-1 deliverable met:** Searchable Vector DB with Metadata — 477 chunks embedded with MedEmbed-large-v0.1, persisted in ChromaDB, retrievable by clinical query with document + section + page citations.

**Next (Day 2):** measure baseline Precision@K on a labeled 15–20 question test set, then compare Top-K values, add BM25 hybrid retrieval, and tune.